In [2]:
from pathlib import Path
from collections import defaultdict
import pandas as pd

# Set root directory (default: current notebook folder)
root_dir = Path.cwd()
if not (root_dir / "Static data").exists():
    raise FileNotFoundError(
        "Static data folder not found. Set root_dir to the dataset folder."
    )

static_dir = root_dir / "Static data"
dynamic_dir = root_dir / "Dynamic data"

# -----------------------------
# 1) Static: device model/manufacturer/brand, Android version, battery fields
# -----------------------------
base_cols = [
    "device_id",
    "device_model",
    "device_manufacturer",
    "device_brand",
    "android_version",
    "android_api",
]

static_files = sorted(static_dir.rglob("*_static*.csv"))
static_rows = []
static_battery_cols = set()

for fp in static_files:
    try:
        df = pd.read_csv(fp, low_memory=False)
    except Exception as e:
        print(f"Skip static file (read error): {fp} -> {e}")
        continue

    # Drop unnamed index-like columns if they exist
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

    battery_cols = [c for c in df.columns if "battery" in c.lower()]
    static_battery_cols.update(battery_cols)

    keep_cols = [c for c in base_cols if c in df.columns] + battery_cols
    if keep_cols:
        static_rows.append(df[keep_cols])

if static_rows:
    static_all = pd.concat(static_rows, ignore_index=True)
else:
    static_all = pd.DataFrame(columns=base_cols)

if "device_id" in static_all.columns:
    static_devices = (
        static_all.drop_duplicates(subset=["device_id"])
        .sort_values("device_id")
        .reset_index(drop=True)
    )
else:
    static_devices = static_all.copy()

print(f"Static files scanned: {len(static_files)}")
print(f"Static battery columns: {sorted(static_battery_cols)}")
print(f"Static devices found: {static_devices['device_id'].nunique() if 'device_id' in static_devices.columns else 0}")

# -----------------------------
# 2) Dynamic: battery-related fields (aggregated per device)
# -----------------------------

def is_dynamic_battery_col(col_name: str) -> bool:
    c = col_name.lower()
    return c.startswith("battery_") or c in {"power_mah"}


def update_stats(stats, device_id, col, series):
    s = series.dropna()
    if s.empty:
        return
    st = stats[device_id][col]
    st["count"] += int(s.count())
    st["sum"] += float(s.sum())
    smin = float(s.min())
    smax = float(s.max())
    st["min"] = smin if st["min"] is None or smin < st["min"] else st["min"]
    st["max"] = smax if st["max"] is None or smax > st["max"] else st["max"]


stats = defaultdict(lambda: defaultdict(lambda: {"count": 0, "sum": 0.0, "min": None, "max": None}))
battery_cols_all = set()

# Scan dynamic files (both processed/combined)
dynamic_files = sorted(dynamic_dir.rglob("*_dynamic*.csv"))
print(f"Dynamic files scanned: {len(dynamic_files)}")

for fp in dynamic_files:
    try:
        header = pd.read_csv(fp, nrows=0).columns.tolist()
    except Exception as e:
        print(f"Skip dynamic file (header error): {fp} -> {e}")
        continue

    if "device_id" not in header:
        continue

    battery_cols = [c for c in header if is_dynamic_battery_col(c)]
    if not battery_cols:
        continue

    battery_cols_all.update(battery_cols)
    usecols = ["device_id"] + battery_cols

    try:
        for chunk in pd.read_csv(fp, usecols=usecols, chunksize=200000, low_memory=False):
            # Convert battery columns to numeric
            for col in battery_cols:
                chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

            for device_id, g in chunk.groupby("device_id"):
                for col in battery_cols:
                    update_stats(stats, device_id, col, g[col])
    except Exception as e:
        print(f"Skip dynamic file (read error): {fp} -> {e}")
        continue

# Build dynamic battery summary per device
rows = []
for device_id, cols in stats.items():
    row = {"device_id": device_id}
    for col, st in cols.items():
        count = st["count"]
        row[f"{col}_count"] = count
        row[f"{col}_min"] = st["min"]
        row[f"{col}_max"] = st["max"]
        row[f"{col}_mean"] = st["sum"] / count if count else None
    rows.append(row)

dynamic_battery_summary = (
    pd.DataFrame(rows).sort_values("device_id").reset_index(drop=True)
    if rows
    else pd.DataFrame(columns=["device_id"])
)

print(f"Dynamic battery columns: {sorted(battery_cols_all)}")
print(f"Dynamic devices found: {dynamic_battery_summary['device_id'].nunique() if 'device_id' in dynamic_battery_summary.columns else 0}")

# -----------------------------
# 3) Merge: device-level info + battery summary
# -----------------------------
if "device_id" in static_devices.columns:
    device_battery_summary = static_devices.merge(
        dynamic_battery_summary, on="device_id", how="left"
    )
else:
    device_battery_summary = dynamic_battery_summary.copy()

# Save outputs
out_dir = root_dir / "outputs"
out_dir.mkdir(exist_ok=True)
static_devices.to_csv(out_dir / "device_static_battery.csv", index=False)
dynamic_battery_summary.to_csv(out_dir / "device_dynamic_battery_summary.csv", index=False)
device_battery_summary.to_csv(out_dir / "device_all_battery_summary.csv", index=False)

print("\nSaved:")
print(out_dir / "device_static_battery.csv")
print(out_dir / "device_dynamic_battery_summary.csv")
print(out_dir / "device_all_battery_summary.csv")

# Preview
static_devices.head(), dynamic_battery_summary.head(), device_battery_summary.head()



Static files scanned: 107
Static battery columns: ['battery_capacity', 'battery_presence', 'battery_scale', 'battery_tech']
Static devices found: 9
Dynamic files scanned: 94
Dynamic battery columns: ['battery_charging_status', 'battery_connection_status', 'battery_current', 'battery_health', 'battery_level', 'battery_power', 'battery_temperature', 'battery_voltage', 'power_mah']
Dynamic devices found: 10

Saved:
d:\DesktopFile\26美赛\26\A dataset from the daily use of features in Android devices\outputs\device_static_battery.csv
d:\DesktopFile\26美赛\26\A dataset from the daily use of features in Android devices\outputs\device_dynamic_battery_summary.csv
d:\DesktopFile\26美赛\26\A dataset from the daily use of features in Android devices\outputs\device_all_battery_summary.csv


(          device_id  device_model device_manufacturer device_brand  \
 0  10270b8780270b80      Moto G60            Motorola     Motorola   
 1  14e6dadbe26a7063  moto g200 5G            motorola     motorola   
 2  263fc77c0785e525    moto g(60)            motorola     motorola   
 3  5d5d4cad78561c3c  Moto G9 Play            Motorola     Motorola   
 4  672d87e6755537a0      SM-G780G             Samsung      Samsung   
 
    android_version  android_api battery_presence  battery_scale battery_tech  \
 0               12           31             True            100       Li-ion   
 1               12           31          present            100       Li-ion   
 2               12           31          present            100       Li-ion   
 3               11           30             True            100       Li-ion   
 4               13           33             True            100       Li-ion   
 
    battery_capacity  
 0            5905.0  
 1            5123.0  
 2            6

In [3]:
# -----------------------------
# 4) Extract target Samsung device files (background/dynamic)
# -----------------------------
from shutil import copy2

target_device_ids = ["672d87e6755537a0", "8f4ce1d51d83c5e3"]

background_dir = root_dir / "Background data"

if not background_dir.exists() or not dynamic_dir.exists():
    raise FileNotFoundError(
        "Background data or Dynamic data folder not found. Set root_dir to the dataset folder."
    )

samsung_dir = root_dir / "Samsung"
background_out_dir = samsung_dir / "Background"
dynamic_out_dir = samsung_dir / "Dynamic"
background_out_dir.mkdir(parents=True, exist_ok=True)
dynamic_out_dir.mkdir(parents=True, exist_ok=True)

def collect_device_files(base_dir, device_ids):
    matched = []
    for fp in base_dir.rglob("*.csv"):
        fp_str = str(fp)
        if any(d in fp_str for d in device_ids):
            matched.append(fp)
    return matched

def copy_device_files(files, base_dir, out_dir):
    copied = []
    for fp in files:
        rel = fp.relative_to(base_dir)
        dest = out_dir / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        copy2(fp, dest)
        copied.append(dest)
    return copied

def count_by_device(files, device_ids):
    counts = {d: 0 for d in device_ids}
    for fp in files:
        fp_str = str(fp)
        for d in device_ids:
            if d in fp_str:
                counts[d] += 1
                break
    return counts

background_files = collect_device_files(background_dir, target_device_ids)
dynamic_files = collect_device_files(dynamic_dir, target_device_ids)

background_copied = copy_device_files(background_files, background_dir, background_out_dir)
dynamic_copied = copy_device_files(dynamic_files, dynamic_dir, dynamic_out_dir)

print("\nSamsung extraction complete.")
print(f"Output folder: {samsung_dir}")
print(f"Background files copied: {len(background_copied)}")
print(f"Dynamic files copied: {len(dynamic_copied)}")
print("Background counts by device_id:", count_by_device(background_files, target_device_ids))
print("Dynamic counts by device_id:", count_by_device(dynamic_files, target_device_ids))

if "device_id" in static_devices.columns:
    target_info = static_devices[static_devices["device_id"].isin(target_device_ids)]
    if not target_info.empty:
        print("\nStatic device info for targets:")
        print(target_info[["device_id", "device_model", "device_brand"]].to_string(index=False))



Samsung extraction complete.
Output folder: d:\DesktopFile\26美赛\26\A dataset from the daily use of features in Android devices\Samsung
Background files copied: 11
Dynamic files copied: 6
Background counts by device_id: {'672d87e6755537a0': 0, '8f4ce1d51d83c5e3': 11}
Dynamic counts by device_id: {'672d87e6755537a0': 5, '8f4ce1d51d83c5e3': 1}

Static device info for targets:
       device_id device_model device_brand
672d87e6755537a0     SM-G780G      Samsung
8f4ce1d51d83c5e3     SM-G780G      samsung


In [5]:
# -----------------------------
# 5) Dynamic feature description (all Dynamic data)
# -----------------------------
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

if "root_dir" not in globals():
    root_dir = Path.cwd()
if "dynamic_dir" not in globals():
    dynamic_dir = root_dir / "Dynamic data"

if not dynamic_dir.exists():
    raise FileNotFoundError(
        "Dynamic data folder not found. Set root_dir to the dataset folder."
    )

dynamic_all_files = sorted(dynamic_dir.rglob("*_dynamic*.csv"))
dynamic_all_cols = set()

for fp in dynamic_all_files:
    try:
        cols = pd.read_csv(fp, nrows=0).columns.tolist()
    except Exception as e:
        print(f"Skip dynamic file (header error): {fp} -> {e}")
        continue
    dynamic_all_cols.update(cols)

battery_cols = sorted(
    [c for c in dynamic_all_cols if c.startswith("battery_")]
    + (["power_mah"] if "power_mah" in dynamic_all_cols else [])
)
frequency_cols = sorted([c for c in dynamic_all_cols if c.startswith("frequency_core")])

feature_groups = [
    (
        "Identity/Time",
        ["device_id", "timestamp", "collected"],
        "Device id and timestamp/collection flag.",
    ),
    (
        "Screen",
        ["screen_status", "bright_level", "bright_mode", "screen_on_time"],
        "Screen state and brightness.",
    ),
    (
        "Connectivity/Hardware",
        [
            "bluetooth",
            "gps_status",
            "gps_activity",
            "saving_mode",
            "nfc",
            "flashlight",
            "airplane_mode",
            "fingerprint",
            "orientation",
        ],
        "Hardware switches and device modes.",
    ),
    (
        "Battery",
        battery_cols,
        "Battery level/health/charging, temperature, current, voltage, power.",
    ),
    (
        "Network",
        [
            "network_mode",
            "mobile_mode",
            "mobile_status",
            "mobile_roaming",
            "mobile_rx",
            "mobile_tx",
            "wifi_status",
            "wifi_intensity",
            "wifi_speed",
            "wifi_ap",
            "wifi_rx",
            "wifi_tx",
            "network_operator",
            "sim_operator",
            "mcc",
            "mnc",
        ],
        "Mobile/WiFi status, signal, traffic, operator info.",
    ),
    (
        "Audio",
        ["ring_mode", "sound_level", "playback_status"],
        "Ringer and audio playback state.",
    ),
    (
        "Memory/Storage",
        ["ram_usage", "ram_free", "rom_usage", "rom_free"],
        "RAM/ROM usage and free space.",
    ),
    (
        "CPU/Thermal",
        ["cpu_usage", "cpu_temperature"] + frequency_cols,
        "CPU usage/temperature and per-core frequency.",
    ),
    (
        "Time usage",
        ["up_time", "sleep_time"],
        "Uptime and sleep time.",
    ),
    (
        "App",
        ["foreground_app"],
        "Foreground app package.",
    ),
]

known_cols = set()
print("\nDynamic feature description (from Dynamic data headers):")
for group, cols, desc in feature_groups:
    present = [c for c in cols if c in dynamic_all_cols]
    if present:
        print(f"- {group}:")
        print(f"  columns: {', '.join(present)}")
        print(f"  meaning: {desc}")
        known_cols.update(present)

extra_cols = sorted([c for c in dynamic_all_cols if c not in known_cols])
if extra_cols:
    print("- Other/Unclassified:")
    print(f"  columns: {', '.join(extra_cols)}")

# -----------------------------
# 6) Export Samsung dynamic battery data and plot
# -----------------------------
if "samsung_dir" not in globals():
    samsung_dir = root_dir / "Samsung"

samsung_dynamic_dir = samsung_dir / "Dynamic"
battery_export_dir = samsung_dir / "BatteryExports"
battery_plot_dir = samsung_dir / "BatteryPlots"
battery_export_dir.mkdir(parents=True, exist_ok=True)
battery_plot_dir.mkdir(parents=True, exist_ok=True)

dyn_files = sorted(samsung_dynamic_dir.rglob("*.csv"))
print(f"\nSamsung dynamic files found: {len(dyn_files)}")

battery_wanted = [
    "device_id",
    "timestamp",
    "battery_level",
    "battery_health",
    "battery_charging_status",
    "battery_connection_status",
    "battery_temperature",
    "battery_current",
    "battery_voltage",
    "battery_power",
    "power_mah",
]


def to_numeric_inplace(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")


for fp in dyn_files:
    try:
        header = pd.read_csv(fp, nrows=0).columns.tolist()
    except Exception as e:
        print(f"Skip dynamic file (header error): {fp} -> {e}")
        continue

    usecols = [c for c in battery_wanted if c in header]
    if "device_id" in header and "device_id" not in usecols:
        usecols.insert(0, "device_id")
    if "timestamp" in header and "timestamp" not in usecols:
        usecols.insert(1, "timestamp")

    if not usecols:
        print(f"Skip (no battery columns): {fp}")
        continue

    try:
        df = pd.read_csv(fp, usecols=usecols, low_memory=False)
    except Exception as e:
        print(f"Skip dynamic file (read error): {fp} -> {e}")
        continue

    # Normalize types
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

    numeric_cols = [c for c in df.columns if c.startswith("battery_") or c == "power_mah"]
    to_numeric_inplace(df, numeric_cols)

    # SOC alias
    if "battery_level" in df.columns:
        df["soc"] = df["battery_level"]

    # Power handling
    power_col = None
    if "battery_power" in df.columns:
        power_col = "battery_power"
    elif "battery_current" in df.columns and "battery_voltage" in df.columns:
        df["battery_power_calc"] = df["battery_current"] * df["battery_voltage"]
        power_col = "battery_power_calc"

    # Export
    export_cols = []
    for c in [
        "device_id",
        "timestamp",
        "soc",
        "battery_level",
        "battery_charging_status",
        "battery_connection_status",
        "battery_health",
        "battery_temperature",
        "battery_current",
        "battery_voltage",
        power_col,
        "power_mah",
    ]:
        if c and c in df.columns:
            export_cols.append(c)

    rel = fp.relative_to(samsung_dynamic_dir)
    out_fp = (battery_export_dir / rel).with_name(rel.stem + "_battery.csv")
    out_fp.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_fp, index=False, columns=export_cols)
    print(f"Exported: {out_fp}")

    # Plot
    if "timestamp" in df.columns and "battery_level" in df.columns and power_col:
        fig, ax1 = plt.subplots(figsize=(10, 4))
        ax1.plot(
            df["timestamp"],
            df["battery_level"],
            color="tab:blue",
            linewidth=1,
            label="battery_level (soc)",
        )
        ax1.set_xlabel("time")
        ax1.set_ylabel("battery_level", color="tab:blue")
        ax1.tick_params(axis="y", labelcolor="tab:blue")

        ax2 = ax1.twinx()
        ax2.plot(
            df["timestamp"],
            df[power_col],
            color="tab:red",
            linewidth=1,
            alpha=0.7,
            label=power_col,
        )
        ax2.set_ylabel(power_col, color="tab:red")
        ax2.tick_params(axis="y", labelcolor="tab:red")

        fig.tight_layout()
        plot_fp = (battery_plot_dir / rel).with_name(rel.stem + "_battery_plot.png")
        plot_fp.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(plot_fp, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"Plot saved: {plot_fp}")
    else:
        print(f"Plot skipped (missing timestamp/battery_level/power): {fp}")

    # Plot current/voltage/temperature vs time on one figure
    cvt_cols = [
        c
        for c in ["battery_current", "battery_voltage", "battery_temperature"]
        if c in df.columns
    ]
    if "timestamp" in df.columns and cvt_cols:
        fig, ax1 = plt.subplots(figsize=(10, 4))
        handles = []
        labels = []

        ax1.set_xlabel("time")

        if "battery_current" in df.columns:
            h1, = ax1.plot(
                df["timestamp"],
                df["battery_current"],
                color="tab:green",
                linewidth=1,
                label="battery_current",
            )
            ax1.set_ylabel("battery_current", color="tab:green")
            ax1.tick_params(axis="y", labelcolor="tab:green")
            handles.append(h1)
            labels.append("battery_current")

        ax2 = None
        if "battery_voltage" in df.columns:
            ax2 = ax1.twinx()
            h2, = ax2.plot(
                df["timestamp"],
                df["battery_voltage"],
                color="tab:purple",
                linewidth=1,
                label="battery_voltage",
            )
            ax2.set_ylabel("battery_voltage", color="tab:purple")
            ax2.tick_params(axis="y", labelcolor="tab:purple")
            handles.append(h2)
            labels.append("battery_voltage")

        ax3 = None
        if "battery_temperature" in df.columns:
            ax3 = ax1.twinx()
            ax3.spines["right"].set_position(("axes", 1.1))
            ax3.set_frame_on(True)
            ax3.patch.set_visible(False)
            h3, = ax3.plot(
                df["timestamp"],
                df["battery_temperature"],
                color="tab:orange",
                linewidth=1,
                label="battery_temperature",
            )
            ax3.set_ylabel("battery_temperature", color="tab:orange")
            ax3.tick_params(axis="y", labelcolor="tab:orange")
            handles.append(h3)
            labels.append("battery_temperature")

        if handles:
            fig.legend(handles, labels, loc="upper left", bbox_to_anchor=(0, 1))

        fig.tight_layout()
        cvt_plot_fp = (battery_plot_dir / rel).with_name(rel.stem + "_battery_cvt_plot.png")
        cvt_plot_fp.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(cvt_plot_fp, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"CVT plot saved: {cvt_plot_fp}")
    else:
        print(f"CVT plot skipped (missing timestamp or CVT columns): {fp}")



Dynamic feature description (from Dynamic data headers):
- Identity/Time:
  columns: device_id, timestamp, collected
  meaning: Device id and timestamp/collection flag.
- Screen:
  columns: screen_status, bright_level, bright_mode, screen_on_time
  meaning: Screen state and brightness.
- Connectivity/Hardware:
  columns: bluetooth, gps_status, gps_activity, saving_mode, nfc, flashlight, airplane_mode, fingerprint, orientation
  meaning: Hardware switches and device modes.
- Battery:
  columns: battery_charging_status, battery_connection_status, battery_current, battery_health, battery_level, battery_power, battery_temperature, battery_voltage, power_mah
  meaning: Battery level/health/charging, temperature, current, voltage, power.
- Network:
  columns: network_mode, mobile_mode, mobile_status, mobile_roaming, mobile_rx, mobile_tx, wifi_status, wifi_intensity, wifi_speed, wifi_ap, wifi_rx, wifi_tx, network_operator, sim_operator, mcc, mnc
  meaning: Mobile/WiFi status, signal, traffic